In [ ]:
from google.colab import files

print("Select ALL of these files at once:")
print("  1. prompt_A_context.txt")
print("  2. prompt_B_generation.txt")
print("  3. prompt_C_diversity.txt")
print("  4. prompt_D_system_role.txt")
print("  5. glorification_seeds.csv")
print()
print("If Ctrl+click multi-select doesn't work, upload them one at a time")
print("by re-running this cell.")
print()

uploaded = files.upload()
print(f"\nUploaded {len(uploaded)} file(s): {list(uploaded.keys())}")

Select ALL of these files at once:
  1. prompt_A_context.txt
  2. prompt_B_generation.txt
  3. prompt_C_diversity.txt
  4. prompt_D_system_role.txt
  5. glorification_seeds.csv

If Ctrl+click multi-select doesn't work, upload them one at a time
by re-running this cell.



Saving glorification_seeds.csv.txt to glorification_seeds.csv.txt
Saving prompt_A_context.txt to prompt_A_context.txt
Saving prompt_B_generation.txt to prompt_B_generation.txt
Saving prompt_C_diversity.txt to prompt_C_diversity.txt
Saving prompt_D_system_role.txt to prompt_D_system_role.txt

Uploaded 5 file(s): ['glorification_seeds.csv.txt', 'prompt_A_context.txt', 'prompt_B_generation.txt', 'prompt_C_diversity.txt', 'prompt_D_system_role.txt']


In [ ]:
!ls -la

total 36
drwxr-xr-x 1 root root 4096 Oct 14 08:03 .
drwxr-xr-x 1 root root 4096 Oct 14 07:53 ..
drwxr-xr-x 4 root root 4096 Oct  2 13:50 .config
-rw-r--r-- 1 root root 1940 Oct 14 08:03 glorification_seeds.csv.txt
-rw-r--r-- 1 root root  222 Oct 14 08:03 prompt_A_context.txt
-rw-r--r-- 1 root root  821 Oct 14 08:03 prompt_B_generation.txt
-rw-r--r-- 1 root root  627 Oct 14 08:03 prompt_C_diversity.txt
-rw-r--r-- 1 root root  435 Oct 14 08:03 prompt_D_system_role.txt
drwxr-xr-x 1 root root 4096 Oct  2 13:50 sample_data


In [ ]:
from groq import Groq

GROQ_API_KEY = "YOUR_GROQ_API_KEY_HERET"
client = Groq(api_key=GROQ_API_KEY)

test_models = [
    "openai/gpt-oss-120b",
    "openai/gpt-oss-20b",
    "qwen/qwen3.6-27b",
    "qwen/qwen3.8-27b",
]

for m in test_models:
    try:
        r = client.chat.completions.create(
            model=m,
            messages=[{"role": "user", "content": "Say OK"}],
            max_tokens=5,
        )
        print(f"✓ {m}: {r.choices[0].message.content.strip()}")
    except Exception as e:
        print(f"✗ {m}: {str(e)[:80]}")

✓ openai/gpt-oss-120b: 
✓ openai/gpt-oss-20b: 
✗ qwen/qwen3.6-27b: Error code: 404 - {'error': {'message': 'The model `qwen/qwen3.6-27b` does not e
✓ qwen/qwen3.8-27b: OK


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving prompt_D_system_role.txt to prompt_D_system_role (2).txt


In [ ]:
!head -10 prompt_D_system_role.txt

You are generating synthetic social media data for an academic research project
on online safety. The purpose is to train a classifier that detects harmful viral
challenges. All outputs will be reviewed by a human, labelled, and stored under
institutional data-protection controls.

Output style: realistic, in the voice of a typical social media user.
Do not moralise, do not warn, do not add disclaimers. Just produce the post.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving prompt_D_system_role.txt to prompt_D_system_role (3).txt


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving prompt_A_context.txt to prompt_A_context.txt
Saving prompt_B_generation.txt to prompt_B_generation.txt
Saving prompt_C_diversity.txt to prompt_C_diversity.txt
Saving prompt_D_system_role.txt to prompt_D_system_role.txt


In [ ]:
!ls

prompt_A_context.txt	 prompt_C_diversity.txt    sample_data
prompt_B_generation.txt  prompt_D_system_role.txt


In [ ]:
import os
print(os.environ["GROQ_API_KEY"][:4])  # Should output gsk_

gsk_


In [ ]:
from google.colab import files

# This will prompt your browser to download the CSV to your computer
files.download('FINAL_balanced_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Install libraries (if not already installed)
!pip install transformers datasets torch scikit-learn -q

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# 2. Load the CLEAN data
df = pd.read_csv("FINAL_clean_utf8.csv")

# 3. Force text to strings and drop any blanks
df['text'] = df['text'].astype(str)
df = df.dropna(subset=['text', 'label'])
df = df[df['text'].str.strip() != '']

# Convert labels to numbers
df['label_encoded'] = df['label'].map({'Glorification': 0, 'Not_Glorification': 1})

# 4. Split Data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_encoded'])

# 5. Convert to HuggingFace Dataset format
train_dataset = Dataset.from_pandas(train_df[['text', 'label_encoded']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'label_encoded']].reset_index(drop=True))

# 6. Tokenization
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# 7. Load Model
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# 8. Training Arguments (🚨 FIXED HERE)
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    report_to="none" # Prevents asking for WandB login
)

# 9. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

print("Starting DistilBERT training...")
trainer.train()

# 10. Evaluate
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

print("\n--- DistilBERT Classification Report ---")
print(classification_report(test_df['label_encoded'], preds, target_names=['Glorification', 'Not_Glorification']))

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

Map:   0%|          | 0/121 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting DistilBERT training...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


ValueError: The model did not return a loss from the inputs, only the following keys: logits. For reference, the inputs it received are input_ids,attention_mask.

In [ ]:
# 1. Install libraries (if not already installed)
!pip install transformers datasets torch scikit-learn -q

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# 2. Load the CLEAN data
df = pd.read_csv("FINAL_clean_utf8.csv")

# 3. Force text to strings and drop any blanks
df['text'] = df['text'].astype(str)
df = df.dropna(subset=['text', 'label'])
df = df[df['text'].str.strip() != '']

# Convert labels to numbers
df['label_encoded'] = df['label'].map({'Glorification': 0, 'Not_Glorification': 1})

# 4. Split Data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_encoded'])

# 5. Convert to HuggingFace Dataset format
# 🛑 FIX: Rename 'label_encoded' to 'labels' for the Trainer
train_df = train_df.rename(columns={'label_encoded': 'labels'})
test_df = test_df.rename(columns={'label_encoded': 'labels'})

train_dataset = Dataset.from_pandas(train_df[['text', 'labels']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'labels']].reset_index(drop=True))

# 6. Tokenization
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# 7. Load Model
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

# 8. Training Arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    report_to="none"
)

# 9. Train
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

print("Starting DistilBERT training...")
trainer.train()

# 10. Evaluate
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

print("\n--- DistilBERT Classification Report ---")
# Note: We use test_df['labels'] here since we renamed the column
print(classification_report(test_df['labels'], preds, target_names=['Glorification', 'Not_Glorification']))

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

Map:   0%|          | 0/121 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting DistilBERT training...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,No log,0.010358
2,No log,0.002818
3,No log,0.002287


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- DistilBERT Classification Report ---
                   precision    recall  f1-score   support

    Glorification       1.00      1.00      1.00        87
Not_Glorification       1.00      1.00      1.00        34

         accuracy                           1.00       121
        macro avg       1.00      1.00      1.00       121
     weighted avg       1.00      1.00      1.00       121



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# 1. Define where to save it in your Drive
save_path = '/content/drive/MyDrive/DistilBERT_Harmful_Content_Model'

# 2. Create the folder if it doesn't exist
os.makedirs(save_path, exist_ok=True)

# 3. Save the model and tokenizer (Assuming 'model' and 'tokenizer' are still in memory)
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ Success! Model and Tokenizer saved to your Google Drive at: {save_path}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Success! Model and Tokenizer saved to your Google Drive at: /content/drive/MyDrive/DistilBERT_Harmful_Content_Model


In [ ]:
# Install the groq library
!pip install groq -q

import os
from groq import Groq

# Set your API key (Replace this with your actual key!)
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"

# Initialize the client
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
print("✅ Groq installed and API key set.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 954.7 kB/s eta 0:00:00
✅ Groq installed and API key set.


In [ ]:
import pandas as pd
import glob
import os

print("Loading and cleaning datasets...")

# 1. Load Glorification Data & Clean Errors
files = sorted(glob.glob("glorification_batch*.csv"))
df_glor = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
# Drop refusals and API errors
bad_phrases = "GENERATION_ERROR|I cannot generate|I can't generate|I'm sorry, but I can't|I will not generate"
df_glor = df_glor[~df_glor['text'].astype(str).str.contains(bad_phrases, case=False, na=False)]
df_glor = df_glor[['text', 'label']]

# 2. Load Awareness Data & Clean
df_aware = pd.read_csv("synthetic_awareness_data.csv")
df_aware = df_aware[~df_aware['text'].astype(str).str.contains(bad_phrases, case=False, na=False)]
# Remove very short incomplete sentences (length < 30)
df_aware = df_aware[df_aware['text'].str.len() > 30]
df_aware = df_aware[['text', 'label']]

# 3. Drop the corrupted Synthetic Participation (We will use Manual data instead)
# (Do not load synthetic_participation_data.csv)

# 4. Load the Manual Excel Data (This is your REAL Participation & Awareness data)
try:
    df_manual = pd.read_excel("challenge_content_collection_1.xlsx", sheet_name="Collection Log")
    df_manual = df_manual.dropna(subset=['Post/comment text', 'Label (your guess)'])
    df_manual = df_manual.rename(columns={'Post/comment text': 'text', 'Label (your guess)': 'label'})
    # Keep only the 3 main classes
    df_manual = df_manual[df_manual['label'].isin(['Awareness', 'Participation', 'Glorification'])]
    df_manual = df_manual[['text', 'label']]
    print(f"✅ Loaded {len(df_manual)} manual rows from Excel (Includes true Participation data).")
except Exception as e:
    print(f"❌ Could not load Excel file: {e}")
    df_manual = pd.DataFrame(columns=['text', 'label'])

# 5. Combine Everything
df_final = pd.concat([df_glor, df_aware, df_manual], ignore_index=True)

# 6. Final Clean & Save
df_final['text'] = df_final['text'].astype(str)
df_final = df_final[df_final['text'].str.strip() != '']
df_final = df_final.drop_duplicates(subset=['text']).sample(frac=1, random_state=42).reset_index(drop=True)

df_final.to_csv("FINAL_3Class_Master_Dataset.csv", index=False, encoding='utf-8')

print("\n=========================================")
print(f"🎉 Dataset Fixed & Saved!")
print(f"Total rows: {len(df_final)}")
print("=========================================")
print("\nFinal Label Distribution:")
print(df_final['label'].value_counts())

Loading and cleaning datasets...
✅ Loaded 32 manual rows from Excel (Includes true Participation data).

🎉 Dataset Fixed & Saved!
Total rows: 250

Final Label Distribution:
label
Awareness        122
Glorification    118
Participation     10
Name: count, dtype: int64


In [ ]:
import pandas as pd
from google.colab import files
import os

try:
    # Check if the file was saved to the Colab folder
    if os.path.exists("synthetic_participation_data.csv"):
        # Load it just to verify it doesn't contain the "I'm sorry" errors
        df_part = pd.read_csv("synthetic_participation_data.csv")

        print(f"✅ Found the file! It contains {len(df_part)} rows.")
        print("Here are the first 2 rows to verify:")
        print(df_part.head(2))

        print("\nTriggering download to your computer...")
        files.download("synthetic_participation_data.csv")

    else:
        print("❌ File not found. Did you run the generation script and save it?")

except Exception as e:
    print(f"An error occurred: {e}")

✅ Found the file! It contains 100 rows.
Here are the first 2 rows to verify:
                                                text          label
0  omg okay so I finally did the blackout challen...  Participation
1  r/BlackoutChallenge\n\n**Day 4: It’s actually ...  Participation

Triggering download to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 1. Install necessary libraries
!pip install transformers datasets torch scikit-learn -q

import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from google.colab import drive
import shutil
from google.colab import files

# 2. Mount Drive
drive.mount('/content/drive')

# 3. Load Data from the Drive path we just found
print("Loading dataset from Drive...")
dataset_path = "/content/drive/MyDrive/Final/FINAL_Master_Dataset.csv"
df = pd.read_csv(dataset_path)

# 4. Map labels (0=Awareness, 1=Glorification, 2=Participation)
label_map = {'Awareness': 0, 'Glorification': 1, 'Participation': 2}
df['label_encoded'] = df['label'].map(label_map)
df = df.dropna(subset=['label_encoded'])
df['label_encoded'] = df['label_encoded'].astype(int)

# 5. Split and Prepare
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_encoded'])
train_df = train_df.rename(columns={'label_encoded': 'labels'})
test_df = test_df.rename(columns={'label_encoded': 'labels'})

train_dataset = Dataset.from_pandas(train_df[['text', 'labels']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['text', 'labels']].reset_index(drop=True))

# 6. Tokenize
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# 7. Train the 3-Class Model
print("Loading DistilBERT and starting training...")
model_3class = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)

training_args = TrainingArguments(
    output_dir='./results_3class_final',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    report_to="none"
)

trainer = Trainer(
    model=model_3class,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

# 8. Evaluate
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
print("\n--- FINAL 3-CLASS CLASSIFICATION REPORT ---")
print(classification_report(test_df['labels'], preds, target_names=['Awareness', 'Glorification', 'Participation']))

# =========================================================
# 🚨 CRITICAL STEP: SAVE AND DOWNLOAD IMMEDIATELY 🚨
# =========================================================
print("\nSaving model to Drive...")
save_path = '/content/drive/MyDrive/DistilBERT_3Class_FINAL'
model_3class.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Model saved to Drive at: {save_path}")

print("Zipping model for download...")
zip_path = '/content/DistilBERT_3Class_FINAL.zip'
shutil.make_archive('/content/DistilBERT_3Class_FINAL', 'zip', save_path)

print("Triggering download...")
files.download(zip_path)
print("✅ Download triggered! Check your Downloads folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading dataset from Drive...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/280 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved to Drive at: /content/drive/MyDrive/DistilBERT_3Class_FINAL
Zipping model for download...
Triggering download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download triggered! Check your Downloads folder.


In [1]:
import torch
import torch.nn.functional as F

# 1. Load the saved model and tokenizer (in case the session restarted)
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from google.colab import drive

drive.mount('/content/drive')
model_path = '/content/drive/MyDrive/DistilBERT_3Class_FINAL'

print("Loading model for testing...")
tokenizer = DistilBertTokenizer.from_pretrained(model_path)
model_3class = DistilBertForSequenceClassification.from_pretrained(model_path)
model_3class.eval()

# 2. Define the testing function
def test_3class_model(text):
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)

    # Predict
    with torch.no_grad():
        outputs = model_3class(**inputs)

    # Get probabilities and predicted class
    probs = F.softmax(outputs.logits, dim=-1)[0]
    pred_idx = torch.argmax(probs).item()

    # Map back to labels
    labels = {0: "Awareness", 1: "Glorification", 2: "Participation"}
    confidence = probs[pred_idx].item() * 100

    print(f"Text: \"{text}\"")
    print(f"Prediction: {labels[pred_idx]} (Confidence: {confidence:.2f}%)\n")

# 3. Run the tests
print("--- TESTING MODEL OUTPUTS ---\n")

# Test 1: Should be Awareness
test_3class_model("Warning: The Momo challenge is a dangerous hoax. Please report any videos you see to the platform.")

# Test 2: Should be Glorification
test_3class_model("Day 5 done! Who else is playing? Drop your level below and join the f57 family 🔥💀")

# Test 3: Should be Participation
test_3class_model("I tried the skull breaker challenge yesterday and my head is still ringing. I don't think I'll do it again.")

Mounted at /content/drive
Loading model for testing...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

--- TESTING MODEL OUTPUTS ---

Text: "Warning: The Momo challenge is a dangerous hoax. Please report any videos you see to the platform."
Prediction: Awareness (Confidence: 89.94%)

Text: "Day 5 done! Who else is playing? Drop your level below and join the f57 family 🔥💀"
Prediction: Glorification (Confidence: 96.84%)

Text: "I tried the skull breaker challenge yesterday and my head is still ringing. I don't think I'll do it again."
Prediction: Participation (Confidence: 94.05%)



In [2]:
!pip install lime -q

import torch
import torch.nn.functional as F
import numpy as np
from lime.lime_text import LimeTextExplainer

# 1. Setup LIME
# Define the labels in the exact order the model learned them
class_names = ['Awareness', 'Glorification', 'Participation']
explainer = LimeTextExplainer(class_names=class_names)

def predict_proba(texts):
    inputs = tokenizer(list(texts), return_tensors="pt", truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        logits = model_3class(**inputs).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
    return probs

# 2. The Complete Risk & Notification Engine
def analyze_post(text):
    print(f"\n{'='*60}")
    print(f"📝 ANALYZING POST: \"{text}\"\n")

    # --- STEP 1: CLASSIFICATION ---
    probs = predict_proba([text])[0]
    pred_idx = np.argmax(probs)
    predicted_intent = class_names[pred_idx]
    confidence = probs[pred_idx] * 100

    print(f"🤖 Model Prediction: {predicted_intent} ({confidence:.2f}% confidence)")

    # --- STEP 2: LIME EXPLAINABILITY ---
    exp = explainer.explain_instance(text, predict_proba, num_features=5, labels=[pred_idx])
    top_words = exp.as_list(label=pred_idx)
    trigger_words = [word for word, weight in top_words if weight > 0]
    print(f"🔍 LIME Trigger Words: {trigger_words}")

    # --- STEP 3: RISK COMPUTATION & NOTIFICATION LOGIC ---
    print("\n🚦 RISK ASSESSMENT & ACTIONS:")

    if predicted_intent == "Awareness":
        risk_level = "Low Risk"
        print(f"   -> Risk Level: {risk_level}")
        print(f"   -> Action: Send **User Alert** (Informational/Educational)")
        print(f"   -> Guardian: Not Notified")

    elif predicted_intent == "Glorification":
        risk_level = "High Risk"
        print(f"   -> Risk Level: {risk_level}")
        print(f"   -> Action: Send **User Alert** (Warning)")
        print(f"   -> Guardian: **Parental Notification Triggered** 🚨 (Reason: Encouraging harmful behavior)")

    elif predicted_intent == "Participation":
        risk_level = "High Risk"
        print(f"   -> Risk Level: {risk_level}")
        print(f"   -> Action: Send **User Alert** (Urgent Help/Resources)")
        print(f"   -> Guardian: **Parental Notification Triggered** 🚨 (Reason: Active self-harm/harmful behavior)")

    print(f"{'='*60}\n")

# 3. Test the Engine!
# Test 1: Awareness (Should be Low Risk, No Guardian)
analyze_post("Warning: The Momo challenge is a dangerous hoax. Please report any videos you see to the platform.")

# Test 2: Glorification (Should be High Risk, Trigger Guardian)
analyze_post("Day 5 done! Who else is playing? Drop your level below and join the f57 family 🔥💀")

# Test 3: Participation (Should be High Risk, Trigger Guardian)
analyze_post("I tried the skull breaker challenge yesterday and my head is still ringing. I don't think I'll do it again.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

📝 ANALYZING POST: "Warning: The Momo challenge is a dangerous hoax. Please report any videos you see to the platform."

🤖 Model Prediction: Awareness (89.94% confidence)
🔍 LIME Trigger Words: [np.str_('hoax'), np.str_('videos'), np.str_('report'), np.str_('Warning')]

🚦 RISK ASSESSMENT & ACTIONS:
   -> Risk Level: Low Risk
   -> Action: Send **User Alert** (Informational/Educational)
   -> Guardian: Not Notified


📝 ANALYZING POST: "Day 5 done! Who else is playing? Drop your level below and join the f57 family 🔥💀"

🤖 Model Prediction: Glorification (96.84% confidence)
🔍 LIME Trigger Words: [np.str_('Day'), np.str_('5'), np.str_('level'), np.str_('done'), np.str_('Drop')]

🚦 RISK ASSESSMENT & ACTIONS:
   -> Risk Level: High Risk
   -> Action: Send **User Alert** (Warning)
   -> Guardian: **Parental Notification Triggered** 🚨 (Reason: Encouraging harmful